In [ ]:
#유튜브 링크 가져오

driver = webdriver.Chrome()

# 1. 유튜브 검색 결과 페이지 접속 (예: 시니어 불편) 많이 반복하기 싶으시면 for 문 사용 추천드립니다
search_query = "넣고 싶은 단어"
url = f"https://www.youtube.com/results?search_query={search_query}" # 유튜브에 검색하기
driver.get(url)
time.sleep(3)

# 2. 링크를 많이 확보하기 위해 스크롤 내리기 (원하는 만큼 반복) 정확성을 위해 과하진 않도록...
for _ in tqdm(range(30)):
    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
    time.sleep(2)

# 3. 영상 링크 요소 찾기
# 유튜브 영상 링크는 id가 'video-title'인 <a> 태그에 들어있습니다.
video_elements = driver.find_elements(By.ID, "video-title")

youtube_link_list = []
for el in video_elements:
    link = el.get_attribute("href")
    # 링크가 None이 아니고, 실제 영상 주소(/watch?v=)인 경우만 저장
    if link and "/watch?v=" in link:
        youtube_link_list.append(link)

# 결과 확인
print(f"수집된 영상 링크 개수: {len(youtube_link_list)}")
for url in youtube_link_list[:5]:
    print(url)

In [ ]:
# 이건 자막 수집은 아니고 유튜브 제목, 내용, 댓글 수집 코드 입니다.
# 수집한 결과를 담을 리스트
youtube_title_list = []
youtube_content_list = []
youtube_reply_list = []

# 앞서 수집한 youtube_link_list가 있다고 가정합니다.
# 테스트를 위해 샘플 리스트를 만듭니다.
# youtube_link_list = ['https://www.youtube.com/watch?v=W1wZWDqpruc'] 

for url in tqdm(youtube_link_list):
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 10)
        time.sleep(3) # 페이지 로딩 대기

        # 1. 영상 제목 (ID: title)
        youtube_title = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.ytd-watch-metadata yt-formatted-string"))).text

        # 2. 본문 내용 (설명란)
        # 설명창이 접혀있는 경우가 많으므로 '더보기'를 눌러 전체 내용을 가져옵니다.
        try:
            expand_btn = driver.find_element(By.ID, "expand")
            expand_btn.click()
            time.sleep(1)
        except:
            pass # 더보기 버튼이 없으면 그냥 진행
        
        youtube_description = driver.find_element(By.ID, "description-inline-expander").text

        # 3. 댓글 수집 (무한 스크롤)
        # 댓글창 로딩을 위해 페이지를 아래로 살짝 내립니다.
        driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
        time.sleep(2)

        # 댓글을 더 많이 보려면 반복 횟수를 늘리세요 (예: range(5))
        for _ in range(3): 
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
            time.sleep(2)

        # 댓글 요소들 수집
        youtube_reply_elements = driver.find_elements(By.ID, "content-text")
        # 댓글들을 하나의 텍스트로 합치기 (줄바꿈 구분)
        youtube_replies_text = "\n".join([c.text for c in youtube_reply_elements])

        # 결과 저장
        youtube_title_list.append(youtube_title)
        youtube_content_list.append(youtube_description)
        youtube_reply_list.append(youtube_replies_text if replies_text else "noun")

    except Exception as e:
        print(f"\n에러 발생 ({url}): {e}")
        continue